## Word 批量生成系统（Excel → Word 模板 → 批注驱动 AI → 审阅回溯）

本 Notebook 按单元格逐步实现并可调试：
- 环境检查与依赖安装
- 目录与测试数据/模板生成
- 元数据加载与校验
- docxtpl 渲染（变量/条件/表格行循环 `{%tr for%}`）
- AI：从**模板** comments 读取 prompt；正文用 `§AIBLOCK0§` 标记段落（docxtpl 会丢弃渲染后的 comments.xml）
- 数据溯源：对渲染结果中与 trace 匹配的 `w:t` 自动加 Word 批注（表名/字段/值）
- **新增：智能标记系统** - 基于位置标记的精确批注生成
- Cell 14：`interactive_smart_generation()` 多模板 ID、确认数据表、批量/单份生成

运行环境：Windows + Jupyter + Python。

# 1、基础环境信息与路径设置

In [1]:
# 基础环境信息与路径设置
import os
import sys
from pathlib import Path

print('Python:', sys.version)
BASE_DIR = Path.cwd()
print('BASE_DIR:', BASE_DIR)

# 统一目录
DIRS = {
    'data': BASE_DIR / 'data',
    'templates': BASE_DIR / 'templates',
    'output': BASE_DIR / 'output',
    'config': BASE_DIR / 'config',
}
DIRS

Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
BASE_DIR: f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system


{'data': WindowsPath('f:/Jupyterworkspace/工行/普惠/word_gen_system/word_gen_system/data'),
 'templates': WindowsPath('f:/Jupyterworkspace/工行/普惠/word_gen_system/word_gen_system/templates'),
 'output': WindowsPath('f:/Jupyterworkspace/工行/普惠/word_gen_system/word_gen_system/output'),
 'config': WindowsPath('f:/Jupyterworkspace/工行/普惠/word_gen_system/word_gen_system/config')}

In [4]:
# Cell 3: 创建目录结构
for k, p in DIRS.items():
    p.mkdir(parents=True, exist_ok=True)
    print(f'OK: {k} -> {p}')

# 2、 数据准备

## 说明：需要准备的 Excel 配置文件

本 Notebook 会自动生成示例文件到 `config/`，你也可以替换为真实业务文件。

1) 实体表（schema）示例：`config/entity_schema.xlsx`
- `table_name`：表名（中文/英文均可）
- `field_name`：字段名（中文/英文均可）
- `is_primary_key`：是否主键（0/1）

2) 关联表（relation）示例：`config/template_relation.xlsx`
- `template_id`：模板 ID（数字）
- `template_name`：模板名称
- `template_file`：模板文件名（位于 `templates/`）
- `table_name`：该模板使用到的实体表
- `role`：`main`/`aux`（`main` 表将绑定到 `data` 别名）

In [40]:
# Cell 4: 生成测试用 Excel 配置文件与业务数据
import pandas as pd
from datetime import date

schema_path = DIRS['config'] / 'entity_schema.xlsx'
relation_path = DIRS['config'] / 'template_relation.xlsx'

# 实体表：声明字段与主键（示例：支行表 + 贷款汇总表 + 产品不良明细表）
schema_df = pd.DataFrame([
    # branch_main（主表：按支行名称生成多份报告）
    {'table_name': 'branch_main', 'field_name': 'branch_name', 'is_primary_key': 1},
    {'table_name': 'branch_main', 'field_name': 'report_date', 'is_primary_key': 0},
    {'table_name': 'branch_main', 'field_name': 'bad_mom', 'is_primary_key': 0},
    {'table_name': 'branch_main', 'field_name': 'bad_yoy', 'is_primary_key': 0},
    # loan_summary（辅表：按支行关联）
    {'table_name': 'loan_summary', 'field_name': 'branch_name', 'is_primary_key': 1},
    {'table_name': 'loan_summary', 'field_name': 'loan_balance', 'is_primary_key': 0},
    {'table_name': 'loan_summary', 'field_name': 'house_eval_amount', 'is_primary_key': 0},
    # products（明细表：一支行多行）
    {'table_name': 'products', 'field_name': 'branch_name', 'is_primary_key': 1},
    {'table_name': 'products', 'field_name': 'product_name', 'is_primary_key': 0},
    {'table_name': 'products', 'field_name': 'overdue_rate', 'is_primary_key': 0},
], columns=['table_name', 'field_name', 'is_primary_key'])

# 关联表：模板需要哪些表，哪个是 main（绑定 data）
relation_df = pd.DataFrame([
    {'template_id': 1, 'template_name': '普惠支行报告', 'template_file': 'template_1.docx', 'table_name': 'branch_main', 'role': 'main'},
    {'template_id': 1, 'template_name': '普惠支行报告', 'template_file': 'template_1.docx', 'table_name': 'loan_summary', 'role': 'aux'},
    {'template_id': 1, 'template_name': '普惠支行报告', 'template_file': 'template_1.docx', 'table_name': 'products', 'role': 'aux'},
], columns=['template_id', 'template_name', 'template_file', 'table_name', 'role'])

with pd.ExcelWriter(schema_path, engine='openpyxl') as w:
    schema_df.to_excel(w, index=False, sheet_name='schema')

with pd.ExcelWriter(relation_path, engine='openpyxl') as w:
    relation_df.to_excel(w, index=False, sheet_name='relation')

print('Wrote:', schema_path)
print('Wrote:', relation_path)

# 业务数据（放到 data/，模拟数据准备文件夹）
branch_main = pd.DataFrame([
    {'branch_name': 'A支行', 'report_date': date(2026, 3, 31), 'bad_mom': 0.12, 'bad_yoy': 0.35},
    {'branch_name': 'B支行', 'report_date': date(2026, 3, 31), 'bad_mom': -0.03, 'bad_yoy': 0.10},
])
loan_summary = pd.DataFrame([
    {'branch_name': 'A支行', 'loan_balance': 1250.0, 'house_eval_amount': 900.0},
    {'branch_name': 'B支行', 'loan_balance': 980.0, 'house_eval_amount': 1200.0},
])
products = pd.DataFrame([
    {'branch_name': 'A支行', 'product_name': '法人e抵', 'overdue_rate': 1.2},
    {'branch_name': 'A支行', 'product_name': '个人e抵押', 'overdue_rate': 0.4},
    {'branch_name': 'B支行', 'product_name': '产品3', 'overdue_rate': 2.5},
])

branch_main.to_excel(DIRS['data'] / 'branch_main.xlsx', index=False)
loan_summary.to_excel(DIRS['data'] / 'loan_summary.xlsx', index=False)
products.to_excel(DIRS['data'] / 'products.xlsx', index=False)

print('Wrote test data into:', DIRS['data'])


## Cell 5 将创建一个示例 Word 模板

模板特点：
- 正文包含 `{{ }}` 变量（来自 `data` 与 `loan_summary`）
- 包含表格循环（products）
- 包含一段文字将被插入批注，批注正文包含 `prompt="...{{data.bad_mom}}..."` 供 AI 使用

注意：模板生成后，你可用 Word 打开并查看批注与模板内容。

In [ ]:
# Cell 5: 生成示例 Word 模板（docxtpl 表格行循环 + 条件 + AI 段标记）
from docx import Document
from docx.shared import Pt

template_path = DIRS['templates'] / 'template_1.docx'

AI_BLOCK_MARKERS = ['§AIBLOCK0§']


def build_demo_template(path: Path) -> None:
    """
    构建演示用 Word 模板。
    表格循环：三行结构 {%tr for %} / 数据行 / {%tr endfor %}。
    """
    doc = Document()
    style = doc.styles['Normal']
    style.font.name = '宋体'
    style.font.size = Pt(11)

    doc.add_heading('普惠支行报告', level=1)
    p = doc.add_paragraph()
    p.add_run('支行：')
    p.add_run('{{ data.branch_name }}')
    p.add_run('，报告期：')
    p.add_run('{{ data.report_date }}')

    p2 = doc.add_paragraph()
    p2.add_run('普惠贷款余额为 ')
    p2.add_run('{{ loan_summary.loan_balance }}')
    p2.add_run(' 万元；房产评估金额 ')
    p2.add_run('{{ loan_summary.house_eval_amount }}')
    p2.add_run(' 万元。')

    doc.add_paragraph(
        '清偿能力判断：{% if loan_summary.house_eval_amount < loan_summary.loan_balance %}无法清偿{% else %}可清偿{% endif %}。'
    )

    doc.add_paragraph('各产品不良情况：')
    table = doc.add_table(rows=4, cols=2)
    table.style = 'Table Grid'
    table.cell(0, 0).text = '产品名称'
    table.cell(0, 1).text = '不良率(%)'
    table.cell(1, 0).text = '{%tr for p in products %}'
    table.cell(1, 1).text = ''
    table.cell(2, 0).text = '{{ p.product_name }}'
    table.cell(2, 1).text = '{{ p.overdue_rate }}'
    table.cell(3, 0).text = '{%tr endfor %}'
    table.cell(3, 1).text = ''

    doc.add_paragraph('风险分析（以下段落将由 AI 改写）：')
    ai_para = doc.add_paragraph(
        '§AIBLOCK0§基于不良环比{{ data.bad_mom }}、同比{{ data.bad_yoy }}，请撰写风险分析与建议。'
    )
    comment_text = (
        'prompt="基于不良额环比{{ data.bad_mom }}、同比{{ data.bad_yoy }}数据，'
        '分析该支行风险状况，给出针对性建议"'
    )
    try:
        doc.add_comment(runs=ai_para.runs, text=comment_text, author='AI', initials='AI')
    except Exception as e:
        raise RuntimeError('python-docx 需 >=1.2 以支持 add_comment') from e

    doc.save(path)


build_demo_template(template_path)
print('Wrote template:', template_path)


# 3、核心功能模块

## 3.1 数据加载模块

In [13]:
# Cell 6: 数据加载模块（增强版，支持中文映射）
import pandas as pd
from typing import Dict, List, Tuple, Any, Optional

def load_entity_schema(schema_xlsx: Path) -> pd.DataFrame:
    """
    加载实体表（表名/字段/主键标记），支持中文映射。

    Args:
        schema_xlsx: config/entity_schema.xlsx 路径
    Returns:
        DataFrame: columns=[table_name, table_name_chinese, field_name, field_name_chinese, is_primary_key]
    Raises:
        FileNotFoundError: 文件不存在
        ValueError: 缺少必要列
    """
    if not schema_xlsx.exists():
        raise FileNotFoundError(f'实体表配置不存在: {schema_xlsx}')
    df = pd.read_excel(schema_xlsx, sheet_name=0)
    required = {'table_name', 'field_name', 'is_primary_key'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'实体表缺少列: {missing}')
    
    # 确保中文列存在，如果不存在则创建空列
    if 'table_name_chinese' not in df.columns:
        df['table_name_chinese'] = ''
    if 'field_name_chinese' not in df.columns:
        df['field_name_chinese'] = ''
    
    return df

def load_template_relation(relation_xlsx: Path) -> pd.DataFrame:
    """
    加载关联表（模板与使用表的多对多关系），支持中文映射。

    Args:
        relation_xlsx: config/template_relation.xlsx 路径
    Returns:
        DataFrame: columns=[template_id, template_name, template_file, table_name, table_name_chinese, role]
    """
    if not relation_xlsx.exists():
        raise FileNotFoundError(f'关联表配置不存在: {relation_xlsx}')
    df = pd.read_excel(relation_xlsx, sheet_name=0)
    required = {'template_id', 'template_name', 'template_file', 'table_name', 'role'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'关联表缺少列: {missing}')
    
    # 确保table_name_chinese列存在，如果不存在则创建空列
    if 'table_name_chinese' not in df.columns:
        df['table_name_chinese'] = ''
    
    return df

def load_data_tables(data_dir: Path, table_names: List[str]) -> Dict[str, pd.DataFrame]:
    """
    从数据准备目录读取指定表名的 Excel 数据。

    Args:
        data_dir: data/ 目录
        table_names: 需要读取的表名列表（将读取 {table}.xlsx）
    Returns:
        dict: {table_name: DataFrame}
    """
    tables = {}
    for t in table_names:
        fp = data_dir / f'{t}.xlsx'
        if not fp.exists():
            raise FileNotFoundError(f'数据表缺失: {fp}（请在 data/ 准备 {t}.xlsx）')
        tables[t] = pd.read_excel(fp)
    return tables

def build_translation_maps(schema_df: pd.DataFrame) -> Dict[str, Dict]:
    """
    从实体表构建中英文翻译映射。

    Args:
        schema_df: load_entity_schema返回的DataFrame
    Returns:
        dict: 包含以下映射的字典：
            - 'table_en_to_cn': {英文表名: 中文表名}
            - 'table_cn_to_en': {中文表名: 英文表名}
            - 'field_en_to_cn': {(英文表名, 英文字段名): 中文字段名}
            - 'field_cn_to_en': {(中文表名, 中文字段名): 英文字段名}
            - 'field_en_to_cn_by_table': {英文表名: {英文字段名: 中文字段名}}
            - 'field_cn_to_en_by_table': {中文表名: {中文字段名: 英文字段名}}
    """
    translation_maps = {
        'table_en_to_cn': {},
        'table_cn_to_en': {},
        'field_en_to_cn': {},
        'field_cn_to_en': {},
        'field_en_to_cn_by_table': {},
        'field_cn_to_en_by_table': {},
    }
    
    # 构建表名映射
    for _, row in schema_df.drop_duplicates('table_name').iterrows():
        en_table = str(row['table_name']).strip()
        cn_table = str(row['table_name_chinese']).strip() if pd.notna(row['table_name_chinese']) else ''
        
        if en_table and cn_table:
            translation_maps['table_en_to_cn'][en_table] = cn_table
            translation_maps['table_cn_to_en'][cn_table] = en_table
    
    # 构建字段名映射
    for _, row in schema_df.iterrows():
        en_table = str(row['table_name']).strip()
        cn_table = str(row['table_name_chinese']).strip() if pd.notna(row['table_name_chinese']) else ''
        en_field = str(row['field_name']).strip()
        cn_field = str(row['field_name_chinese']).strip() if pd.notna(row['field_name_chinese']) else ''
        
        if en_table and en_field and cn_field:
            translation_maps['field_en_to_cn'][(en_table, en_field)] = cn_field
            if cn_table:
                translation_maps['field_cn_to_en'][(cn_table, cn_field)] = en_field
            
            # 按表组织的映射
            if en_table not in translation_maps['field_en_to_cn_by_table']:
                translation_maps['field_en_to_cn_by_table'][en_table] = {}
            translation_maps['field_en_to_cn_by_table'][en_table][en_field] = cn_field
            
            if cn_table and cn_table not in translation_maps['field_cn_to_en_by_table']:
                translation_maps['field_cn_to_en_by_table'][cn_table] = {}
            if cn_table and cn_field:
                translation_maps['field_cn_to_en_by_table'][cn_table][cn_field] = en_field
    
    return translation_maps

def translate_table_name(table_name: str, translation_maps: Dict[str, Dict], 
                        to_chinese: bool = True) -> str:
    """
    翻译表名（中英文互译）。

    Args:
        table_name: 要翻译的表名
        translation_maps: build_translation_maps返回的映射字典
        to_chinese: True表示英文->中文，False表示中文->英文
    Returns:
        翻译后的表名，如果找不到映射则返回原表名
    """
    if to_chinese:
        return translation_maps['table_en_to_cn'].get(table_name, table_name)
    else:
        return translation_maps['table_cn_to_en'].get(table_name, table_name)

def translate_field_name(table_name: str, field_name: str, 
                        translation_maps: Dict[str, Dict],
                        to_chinese: bool = True) -> str:
    """
    翻译字段名（中英文互译）。

    Args:
        table_name: 表名（英文或中文）
        field_name: 字段名（英文或中文）
        translation_maps: build_translation_maps返回的映射字典
        to_chinese: True表示英文->中文，False表示中文->英文
    Returns:
        翻译后的字段名，如果找不到映射则返回原字段名
    """
    if to_chinese:
        # 先尝试英文表名+英文字段名
        result = translation_maps['field_en_to_cn'].get((table_name, field_name), field_name)
        if result == field_name:
            # 如果直接查找失败，可能表名已经是中文，需要先翻译表名
            en_table = translation_maps['table_cn_to_en'].get(table_name, table_name)
            result = translation_maps['field_en_to_cn'].get((en_table, field_name), field_name)
        return result
    else:
        # 先尝试中文表名+中文字段名
        result = translation_maps['field_cn_to_en'].get((table_name, field_name), field_name)
        if result == field_name:
            # 如果直接查找失败，可能表名已经是英文，需要先翻译表名
            cn_table = translation_maps['table_en_to_cn'].get(table_name, table_name)
            result = translation_maps['field_cn_to_en'].get((cn_table, field_name), field_name)
        return result

# --- 测试 ---
schema_df2 = load_entity_schema(DIRS['config'] / 'entity_schema.xlsx')
relation_df2 = load_template_relation(DIRS['config'] / 'template_relation.xlsx')
print('实体表结构:')
print(schema_df2.head())
print('\n关联表结构:')
print(relation_df2.head())

# 构建映射
translation_maps = build_translation_maps(schema_df2)
print('\n表名映射示例:')
for en, cn in list(translation_maps['table_en_to_cn'].items())[:3]:
    print(f'  {en} -> {cn}')
print('\n字段映射示例:')
for (table, field), cn in list(translation_maps['field_en_to_cn'].items())[:3]:
    print(f'  {table}.{field} -> {cn}')

实体表结构:
     table_name table_name_chinese   field_name field_name_chinese  \
0   branch_main               支行主表  branch_name               支行名称   
1   branch_main               支行主表  report_date               报告日期   
2   branch_main               支行主表      bad_mom             不良率月环比   
3   branch_main               支行主表      bad_yoy              不良率同比   
4  loan_summary              贷款汇总表  branch_name               支行名称   

   is_primary_key  
0               1  
1               0  
2               0  
3               0  
4               1  

关联表结构:
   template_id template_name    template_file      table_name  \
0            1        普惠支行报告  template_1.docx     branch_main   
1            1        普惠支行报告  template_1.docx    loan_summary   
2            1        普惠支行报告  template_1.docx        products   
3            2    法人e抵单面签署材料  template_2.docx  loan_cust_data   
4            3    法人e抵双面签署材料  template_3.docx  loan_cust_data   

  table_name_chinese  role  
0               支行主表  ma

## 3.2 模板选择与依赖表确认

In [14]:
# Cell 7: 模板选择与依赖表确认（模拟 CLI）
def list_templates(relation_df: pd.DataFrame) -> pd.DataFrame:
    """
    列出模板列表（按 template_id 唯一）。

    Args:
        relation_df: 关联表 DataFrame
    Returns:
        DataFrame: 模板列表
    """
    return (relation_df[['template_id', 'template_name', 'template_file']]
            .drop_duplicates()
            .sort_values('template_id'))

def get_template_tables(relation_df: pd.DataFrame, template_id: int, 
                       translation_maps: Optional[Dict[str, Dict]] = None) -> Tuple[str, List[str], str]:
    """
    获取某模板所需表名列表，并返回 main 表名（支持中文映射）。
    
    Args:
        relation_df: 关联表
        template_id: 模板ID
        translation_maps: 翻译映射字典（可选），如果不提供则直接使用表名字段
    Returns:
        (template_file, table_names, main_table)
    """
    sub = relation_df[relation_df['template_id'] == template_id]
    if sub.empty:
        raise ValueError(f'未找到 template_id={template_id} 的配置')
    
    # 获取模板文件（支持中文文件名）
    template_file = sub['template_file'].iloc[0]
    
    # 获取表名列表（如果提供了翻译映射，优先使用中文表名）
    table_names = []
    for _, row in sub.iterrows():
        table_name = str(row['table_name']).strip()
        table_name_chinese = str(row['table_name_chinese']).strip() if pd.notna(row.get('table_name_chinese', '')) else ''
        
        # 如果提供了翻译映射，优先使用中文表名
        if translation_maps and table_name_chinese:
            # 检查中文表名是否在映射中
            en_table = translation_maps['table_cn_to_en'].get(table_name_chinese)
            if en_table:
                table_names.append(en_table)
            else:
                table_names.append(table_name)
        else:
            table_names.append(table_name)
    
    # 去重
    table_names = list(set(table_names))
    
    # 获取主表
    main_rows = sub[sub['role'].astype(str).str.lower() == 'main']
    if len(main_rows) != 1:
        raise ValueError('关联表中 role=main 必须且只能有一条，用于绑定 data 别名')
    
    main_table = main_rows['table_name'].iloc[0]
    
    # 如果提供了翻译映射，并且主表有中文名，检查是否需要使用中文名
    if translation_maps:
        main_table_chinese = str(main_rows['table_name_chinese'].iloc[0]).strip() if 'table_name_chinese' in main_rows.columns and pd.notna(main_rows['table_name_chinese'].iloc[0]) else ''
        if main_table_chinese:
            # 检查中文表名是否在映射中
            en_main_table = translation_maps['table_cn_to_en'].get(main_table_chinese)
            if en_main_table:
                main_table = en_main_table
    
    return template_file, table_names, main_table

## 3.3 构建上下文（按主键生成多份报告实例） + TraceMap

In [15]:
# Cell 8: 构建上下文（按主表主键：多行批量 / 单行单份）+ TraceMap + 条件表达式追踪
from dataclasses import dataclass
from typing import Optional, Any, Dict, List, Tuple
from datetime import date, datetime
import re
from pathlib import Path
import zipfile
from lxml import etree

@dataclass
class TraceItem:
    """
    数据来源追溯项（用于审阅批注）。

    Attributes:
        var: 模板侧变量路径，如 data.branch_name、products.0.product_name
        table: Excel 实体表名
        field: 字段名
        value: 原始值
        pk: 当前报告实例主键值
        row_index: 数据行索引（可选）
        source_file: 数据文件名（可选）
    """
    var: str
    table: str
    field: str
    value: Any
    pk: Any
    row_index: Optional[int] = None
    source_file: Optional[str] = None


def format_display_value(v: Any) -> str:
    """格式化为 Word 中常见的纯文本，便于与 w:t 精确匹配。"""
    if v is None:
        return ''
    if isinstance(v, datetime):
        return v.strftime('%Y-%m-%d %H:%M:%S')
    if isinstance(v, date):
        return v.strftime('%Y-%m-%d')
    if isinstance(v, bool):
        return 'True' if v else 'False'
    if isinstance(v, float):
        if abs(v - round(v, 1)) < 1e-9:
            s = str(round(v, 1))
            return s[:-2] if s.endswith('.0') else s
        if abs(v - int(v)) < 1e-9:
            return str(int(v))
    if isinstance(v, int):
        return str(v)
    return str(v).strip()


def get_primary_key_field(schema_df: pd.DataFrame, table_name: str) -> str:
    sub = schema_df[(schema_df['table_name'] == table_name) & (schema_df['is_primary_key'] == 1)]
    if sub.empty:
        raise ValueError(f'表 {table_name} 未配置主键字段')
    if len(sub) != 1:
        raise ValueError(f"表 {table_name} 暂仅支持单主键，当前: {sub['field_name'].tolist()}")
    return sub['field_name'].iloc[0]


# ========== 条件表达式追踪功能（原本在 condition_tracker.py）==========
def extract_condition_expressions_from_docx(template_path: Path) -> List[Tuple[str, str, str]]:
    """
    从Word模板中提取所有条件表达式。
    
    返回: [(condition_expr, true_text, false_text), ...]
    """
    expressions = []
    
    try:
        # 读取Word文档的document.xml
        with zipfile.ZipFile(template_path, 'r') as z:
            xml_content = z.read('word/document.xml').decode('utf-8')
        
        # 查找所有段落文本中的条件表达式
        pattern = r'\{%\s*if\s+(.+?)\s*%\}(.*?)(?:\{%\s*else\s*%\}(.*?))?\{%\s*endif\s*%\}'        
        for match in re.finditer(pattern, xml_content, re.DOTALL):
            condition = match.group(1).strip()
            true_text = match.group(2).strip() if match.group(2) else ""
            false_text = match.group(3).strip() if match.group(3) else ""
            
            # 清理文本中的XML标签和多余空白
            true_text = re.sub(r'<[^>]+>', '', true_text)
            false_text = re.sub(r'<[^>]+>', '', false_text)
            true_text = ' '.join(true_text.split())
            false_text = ' '.join(false_text.split())
            
            expressions.append((condition, true_text, false_text))
            
    except Exception as e:
        print(f"警告: 提取条件表达式时出错: {e}")
    
    return expressions


def extract_condition_expressions_from_text(template_text: str) -> List[Tuple[str, str, str]]:
    """
    从文本中提取所有条件表达式。
    
    返回: [(condition_expr, true_text, false_text), ...]
    """
    expressions = []
    
    try:
        # 匹配 {% if ... %}...{% else %}...{% endif %} 结构
        pattern = r'\{%\s*if\s+(.+?)\s*%\}(.*?)(?:\{%\s*else\s*%\}(.*?))?\{%\s*endif\s*%\}'
        
        for match in re.finditer(pattern, template_text, re.DOTALL):
            condition = match.group(1).strip()
            true_text = match.group(2).strip() if match.group(2) else ""
            false_text = match.group(3).strip() if match.group(3) else ""
            
            # 清理文本
            true_text = ' '.join(true_text.split())
            false_text = ' '.join(false_text.split())
            
            expressions.append((condition, true_text, false_text))
            
    except Exception as e:
        print(f"警告: 提取条件表达式时出错: {e}")
    
    return expressions


def evaluate_condition_expression(expression: str, context: Dict[str, Any]) -> Tuple[bool, Dict[str, Any]]:
    """
    安全地计算条件表达式的结果。
    
    返回: (结果, 表达式解析信息字典)
    """
    try:
        # 简单解析表达式，提取变量和操作符
        expr_info = {
            'expression': expression,
            'variables': [],
            'operators': [],
            'raw_result': None,
            'error': None
        }
        
        # 提取变量名
        variable_pattern = r'[a-zA-Z_][a-zA-Z0-9_.]*'
        variables = re.findall(variable_pattern, expression)
        expr_info['variables'] = list(set(variables))
        
        # 创建安全的求值环境
        safe_globals = {
            '__builtins__': {},
            'True': True,
            'False': False,
            'None': None,
            'abs': abs,
            'int': int,
            'float': float,
            'str': str,
            'bool': bool,
            'len': len,
            'sum': sum,
            'min': min,
            'max': max,
            'round': round,
        }
        
        # 创建一个访问器函数来安全地获取嵌套值
        def get_nested_value(obj, path):
            """安全地获取嵌套字典/列表中的值"""
            parts = path.split('.')
            value = obj
            for part in parts:
                if isinstance(value, dict) and part in value:
                    value = value[part]
                elif isinstance(value, list) and part.isdigit() and int(part) < len(value):
                    value = value[int(part)]
                elif hasattr(value, part):
                    value = getattr(value, part)
                else:
                    return None
            return value
        
        # 准备求值环境
        safe_locals = {'context': context, 'get_nested_value': get_nested_value}
        
        # 为表达式中的每个变量创建访问器
        for var_name in variables:
            if '.' in var_name:
                # 为嵌套变量创建访问器
                parts = var_name.split('.')
                value = get_nested_value(context, var_name)
                safe_locals[var_name] = value
            elif var_name in context:
                safe_locals[var_name] = context[var_name]
            else:
                safe_locals[var_name] = None
        
        # 尝试直接求值
        try:
            result = eval(expression, safe_globals, safe_locals)
            expr_info['raw_result'] = result
            return bool(result), expr_info
        except Exception as e:
            expr_info['error'] = str(e)
            
            # 如果直接求值失败，尝试替换表达式中的变量
            expr_for_eval = expression
            for var_name in variables:
                if var_name in safe_locals and safe_locals[var_name] is not None:
                    value = safe_locals[var_name]
                    if isinstance(value, str):
                        replacement = repr(value)
                    else:
                        replacement = str(value)
                    expr_for_eval = expr_for_eval.replace(var_name, replacement)
            
            # 尝试求值替换后的表达式
            try:
                result = eval(expr_for_eval, safe_globals, {})
                expr_info['raw_result'] = result
                expr_info['evaluated_expression'] = expr_for_eval
                return bool(result), expr_info
            except Exception as e2:
                expr_info['error'] = f"原始错误: {e}, 替换后错误: {e2}"
            
            # 如果eval失败，尝试简单的比较解析
            comparison_patterns = [
                (r'(.+?)\s*<\s*(.+)', 'lt'),
                (r'(.+?)\s*<=\s*(.+)', 'le'),
                (r'(.+?)\s*>\s*(.+)', 'gt'),
                (r'(.+?)\s*>=\s*(.+)', 'ge'),
                (r'(.+?)\s*==\s*(.+)', 'eq'),
                (r'(.+?)\s*!=\s*(.+)', 'ne'),
                (r'(.+?)\s+in\s+(.+)', 'in'),
                (r'(.+?)\s+not\s+in\s+(.+)', 'not_in'),
            ]
            
            for pattern, op_type in comparison_patterns:
                match = re.match(pattern, expression.strip())
                if match:
                    left, right = match.group(1), match.group(2)
                    try:
                        left_val = eval(left.strip(), safe_globals, safe_locals) if '.' in left.strip() or left.strip() in safe_locals else float(left.strip())
                        right_val = eval(right.strip(), safe_globals, safe_locals) if '.' in right.strip() or right.strip() in safe_locals else float(right.strip())
                        
                        if op_type == 'lt':
                            result = left_val < right_val
                        elif op_type == 'le':
                            result = left_val <= right_val
                        elif op_type == 'gt':
                            result = left_val > right_val
                        elif op_type == 'ge':
                            result = left_val >= right_val
                        elif op_type == 'eq':
                            result = left_val == right_val
                        elif op_type == 'ne':
                            result = left_val != right_val
                        elif op_type == 'in':
                            result = left_val in right_val
                        elif op_type == 'not_in':
                            result = left_val not in right_val
                        else:
                            result = False
                        
                        expr_info['raw_result'] = result
                        expr_info['parsed'] = {
                            'left': left,
                            'right': right,
                            'operator': op_type,
                            'left_value': left_val,
                            'right_value': right_val
                        }
                        return bool(result), expr_info
                    except:
                        continue
            
            # 如果所有方法都失败，返回False
            return False, expr_info
            
    except Exception as e:
        print(f"警告: 计算条件表达式 '{expression}' 时出错: {e}")
        return False, {'expression': expression, 'error': str(e)}


def create_trace_item_from_condition(
    condition: str,
    result: bool,
    display_text: str,
    expr_info: Dict[str, Any],
    pk_val: Any,
    true_text: str,
    false_text: str
) -> Dict[str, Any]:
    """
    从条件表达式创建TraceItem兼容的溯源项字典。
    返回的字典具有与TraceItem相同的字段，外加额外的条件信息。
    """
    return {
        'var': f'条件表达式: {condition}',
        'table': '__condition__',
        'field': condition,
        'value': display_text,
        'pk': pk_val,
        'row_index': None,
        'source_file': 'template_condition',
        # 额外字段用于条件表达式
        '_condition_result': result,
        '_expression_info': expr_info,
        '_true_text': true_text,
        '_false_text': false_text,
        '_is_condition': True  # 标记这是一个条件表达式项
    }


def add_condition_traces(
    context: Dict[str, Any],
    trace_map: Dict[str, Any],
    pk_val: Any,
    template_path: Path = None,
    template_text: str = None
) -> None:
    """
    为模板中的条件表达式添加溯源项。
    
    参数:
        context: 渲染上下文
        trace_map: 当前的trace映射（将接受TraceItem或ConditionTraceItem）
        pk_val: 主键值
        template_path: Word模板路径
        template_text: 模板文本（如果template_path未提供）
    """
    # 提取条件表达式
    expressions = []
    if template_path and template_path.exists():
        expressions = extract_condition_expressions_from_docx(template_path)
    elif template_text:
        expressions = extract_condition_expressions_from_text(template_text)
    else:
        # 如果没有模板信息，尝试从上下文推断常见表达式
        expressions = [
            ('loan_summary.house_eval_amount < loan_summary.loan_balance', '无法清偿', '可清偿')
        ]
    
    # 计算每个表达式并添加到trace
    for condition, true_text, false_text in expressions:
        result, expr_info = evaluate_condition_expression(condition, context)
        
        # 确定显示的文本
        display_text = true_text if result else false_text
        
        if display_text:
            # 创建ConditionTraceItem
            trace_item = create_trace_item_from_condition(
                condition=condition,
                result=result,
                display_text=display_text,
                expr_info=expr_info,
                pk_val=pk_val,
                true_text=true_text,
                false_text=false_text
            )
            
            # 添加到trace映射
            trace_key = f'__condition__.{hash(condition) & 0xFFFFFFFF}'
            trace_map[trace_key] = trace_item
            
            # 同时为显示文本创建标准TraceItem兼容项（用于批注匹配）
            literal_key = f'__literal__.{hash(display_text) & 0xFFFFFFFF}'
            trace_map[literal_key] = trace_item


def build_report_contexts(
    schema_df: pd.DataFrame,
    tables: Dict[str, pd.DataFrame],
    main_table: str,
    alias_map: Dict[str, str] | None = None,
    template_path: Path | None = None,
    template_text: str | None = None,
    translation_maps: Optional[Dict[str, Dict]] = None,
) -> List[Tuple[dict, Dict[str, Any]]]:
    """
    按主表主键逐行生成报告；主表一行则仅一份。
    辅表：与主键同名列则过滤；单行 dict，多行 list；Trace 含 products.i.field。
    
    新增参数:
        template_path: Word模板路径（用于提取条件表达式）
        template_text: 模板文本（如果template_path未提供）
        translation_maps: 中英文翻译映射字典（如果提供，将创建中文别名）
    """
    if alias_map is None:
        alias_map = {t: t for t in tables.keys()}

    pk_field = get_primary_key_field(schema_df, main_table)
    if pk_field not in tables[main_table].columns:
        raise ValueError(f'主表 {main_table} 缺少主键列 {pk_field}')

    out: List[Tuple[dict, Dict[str, Any]]] = []
    for idx, row in tables[main_table].iterrows():
        pk_val = row[pk_field]
        context: dict = {}
        trace: Dict[str, Any] = {}

        # 处理主表数据（data别名）
        main_data = row.to_dict()
        context['data'] = main_data
        
        # 如果提供了翻译映射，并且主表有中文名，也创建中文data别名
        main_table_cn = None
        if translation_maps:
            main_table_cn = translation_maps['table_en_to_cn'].get(main_table)
            if main_table_cn:
                # 创建中文字段名的主表数据
                cn_main_data = {}
                field_map = translation_maps['field_en_to_cn_by_table'].get(main_table, {})
                for en_field, cn_field in field_map.items():
                    if en_field in main_data:
                        cn_main_data[cn_field] = main_data[en_field]
                # 如果创建了中文数据，添加到上下文
                if cn_main_data:
                    context[main_table_cn] = cn_main_data
        
        # 为主表每个字段创建trace
        for k, v in main_data.items():
            var = f'data.{k}'
            trace[var] = TraceItem(
                var=var, table=main_table, field=k, value=v, pk=pk_val,
                row_index=int(idx), source_file=f'{main_table}.xlsx',
            )

        for tname, df in tables.items():
            if tname == main_table:
                continue
            alias = alias_map.get(tname, tname)
            sub = df[df[pk_field] == pk_val] if pk_field in df.columns else df

            if len(sub) == 1:
                obj = sub.iloc[0].to_dict()
                context[alias] = obj
                ri = int(sub.index[0])
                for k, v in obj.items():
                    var = f'{alias}.{k}'
                    trace[var] = TraceItem(
                        var=var, table=tname, field=k, value=v, pk=pk_val,
                        row_index=ri, source_file=f'{tname}.xlsx',
                    )
                
                # 如果提供了翻译映射，并且该表有中文名，创建中文别名
                if translation_maps:
                    table_cn = translation_maps['table_en_to_cn'].get(tname)
                    if table_cn:
                        # 创建中文字段名的数据
                        cn_obj = {}
                        field_map = translation_maps['field_en_to_cn_by_table'].get(tname, {})
                        for en_field, cn_field in field_map.items():
                            if en_field in obj:
                                cn_obj[cn_field] = obj[en_field]
                        # 如果创建了中文数据，添加到上下文
                        if cn_obj:
                            context[table_cn] = cn_obj
                        
                        # 为中文字段创建trace（指向相同的值和源表）
                        for en_field, cn_field in field_map.items():
                            if en_field in obj:
                                var_cn = f'{table_cn}.{cn_field}'
                                trace[var_cn] = TraceItem(
                                    var=var_cn, table=tname, field=en_field, value=obj[en_field], pk=pk_val,
                                    row_index=ri, source_file=f'{tname}.xlsx',
                                )
            else:
                lst = sub.to_dict(orient='records')
                context[alias] = lst
                
                # 如果提供了翻译映射，并且该表有中文名，创建中文别名列表
                cn_lst = None
                field_map = None
                if translation_maps:
                    table_cn = translation_maps['table_en_to_cn'].get(tname)
                    if table_cn:
                        field_map = translation_maps['field_en_to_cn_by_table'].get(tname, {})
                        if field_map:
                            cn_lst = []
                            for rec in lst:
                                cn_rec = {}
                                for en_field, cn_field in field_map.items():
                                    if en_field in rec:
                                        cn_rec[cn_field] = rec[en_field]
                                cn_lst.append(cn_rec)
                            if cn_lst:
                                context[table_cn] = cn_lst
                
                for row_i, rec in enumerate(lst):
                    ri = int(sub.index[row_i]) if row_i < len(sub.index) else None
                    for k, v in rec.items():
                        var = f'{alias}.{row_i}.{k}'
                        trace[var] = TraceItem(
                            var=var, table=tname, field=k, value=v, pk=pk_val,
                            row_index=ri, source_file=f'{tname}.xlsx',
                        )
                    
                    # 为中文字段创建trace（如果存在）
                    if cn_lst and field_map and row_i < len(cn_lst):
                        cn_rec = cn_lst[row_i]
                        for en_field, cn_field in field_map.items():
                            if en_field in rec:
                                var_cn = f'{table_cn}.{row_i}.{cn_field}'
                                trace[var_cn] = TraceItem(
                                    var=var_cn, table=tname, field=en_field, value=rec[en_field], pk=pk_val,
                                    row_index=ri, source_file=f'{tname}.xlsx',
                                )

        # 添加条件表达式溯源（通用，支持任意条件表达式）
        add_condition_traces(
            context=context,
            trace_map=trace,
            pk_val=pk_val,
            template_path=template_path,
            template_text=template_text
        )
        
        out.append((context, trace))

    return out


# --- 测试 ---
template_file, table_names, main_table = get_template_tables(relation_df2, template_id=1)
tables_loaded = load_data_tables(DIRS['data'], table_names)
contexts = build_report_contexts(schema_df2, tables_loaded, main_table=main_table)
print('报告份数（=主表行数）:', len(contexts))
print('首份 trace 条数:', len(contexts[0][1]))


报告份数（=主表行数）: 2
首份 trace 条数: 15


## 3.3.1 智能标记系统（新增）

In [16]:
# Cell 8.1: 智能标记系统 - 基于位置标记的精确批注生成
# 导入智能标记系统
from smart_marking_system import (
    TemplateMarker, SmartAnnotationGenerator, MarkerType,
    PositionMarker, DocumentPosition, PositionMapping,
    trace_filter, create_trace_mapping
)

# 增强的模板渲染函数 - 支持智能标记
def render_docx_template_with_markers(
    template_docx: Path, 
    context: dict, 
    output_docx: Path,
    trace_map: Dict[str, Any],
    use_smart_marking: bool = True
) -> Optional[Dict[str, PositionMapping]]:
    """
    使用智能标记系统渲染Word模板
    
    Args:
        template_docx: 模板文件路径
        context: 渲染上下文
        output_docx: 输出文件路径
        trace_map: 溯源映射
        use_smart_marking: 是否使用智能标记
        
    Returns:
        位置映射字典（如果使用智能标记），否则None
    """
    if not use_smart_marking:
        # 使用原渲染方式
        from docxtpl import DocxTemplate
        doc = DocxTemplate(str(template_docx))
        doc.render(context)
        doc.save(str(output_docx))
        return None
    
    # 读取模板内容
    from docx import Document
    doc = Document(template_docx)
    
    # 提取模板文本（简化版本，实际应更复杂）
    template_text = ""
    for para in doc.paragraphs:
        template_text += para.text + "\n"
    
    # 标记模板
    marker = TemplateMarker()
    marked_template, markers = marker.mark_template(template_text)
    
    # 创建Jinja2环境并注册trace过滤器
    from jinja2 import Environment, StrictUndefined
    env = Environment(undefined=StrictUndefined)
    env.filters['trace'] = trace_filter
    
    # 渲染标记后的模板（简化版本）
    # 实际实现需要更复杂的逻辑来保持Word格式
    rendered_text = env.from_string(marked_template).render(context)
    
    # 创建位置映射（简化版本）
    position_mappings = {}
    for marker_obj in markers:
        # 查找对应的TraceItem
        trace_item = None
        if marker_obj.marker_type == MarkerType.VARIABLE:
            for key, item in trace_map.items():
                var_path = marker_obj.var_path
                if (hasattr(item, 'var') and item.var == var_path) or \
                   (isinstance(item, dict) and item.get('var') == var_path):
                    trace_item = item
                    break
        
        if trace_item:
            # 创建文档位置（简化版本）
            doc_position = DocumentPosition(
                marker_id=marker_obj.marker_id,
                xml_path=f"/dummy/path/for/{marker_obj.marker_id}",
                text_range=(0, 0),  # 实际应计算
                text_content=str(trace_item.value if hasattr(trace_item, 'value') else trace_item.get('value', '')),
                element_ref=None
            )
            
            position_mappings[marker_obj.marker_id] = PositionMapping(
                marker=marker_obj,
                doc_position=doc_position,
                trace_item=trace_item
            )
    
    # 保存渲染后的文档（简化版本）
    from docx import Document as NewDocument
    new_doc = NewDocument()
    new_doc.add_paragraph(rendered_text)
    new_doc.save(output_docx)
    
    return position_mappings


# 增强的批注生成函数
def annotate_values_with_smart_marking(
    doc_root,
    comments_root,
    trace_map: Dict[str, Any],
    position_mappings: Optional[Dict[str, PositionMapping]] = None
) -> None:
    """
    智能批注生成：优先使用位置映射，回退到文本匹配
    """
    if position_mappings:
        # 使用智能标记系统
        generator = SmartAnnotationGenerator()
        mappings_list = list(position_mappings.values())
        generator.annotate_by_mapping(doc_root, mappings_list, comments_root)
    else:
        # 回退到原文本匹配方式
        annotate_values_in_document(doc_root, comments_root, trace_map)


# 增强的最终文档处理函数
def finalize_rendered_document_enhanced(
    rendered_docx: Path,
    template_docx: Path,
    context: dict,
    trace_map: Dict[str, TraceItem],
    output_docx: Path,
    use_ai: bool,
    api_key: str | None,
    ai_markers: List[str] | None = None,
    use_smart_marking: bool = True,
    position_mappings: Optional[Dict[str, PositionMapping]] = None
) -> Path:
    """
    增强的最终文档处理：支持智能标记
    """
    if ai_markers is None:
        ai_markers = globals().get('AI_BLOCK_MARKERS', ['§AIBLOCK0§'])

    # 加载文件
    import io
    import zipfile
    from lxml import etree
    
    W_NS = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
    
    def _load_zip_dict(docx_path: Path) -> dict:
        with zipfile.ZipFile(docx_path, 'r') as z:
            return {n: z.read(n) for n in z.namelist()}
            
    def _ensure_comments_infra(files: dict):
        # 简化实现
        CT_NS = 'http://schemas.openxmlformats.org/package/2006/content-types'
        REL_NS = 'http://schemas.openxmlformats.org/package/2006/relationships'
        
        ct = etree.fromstring(files['[Content_Types].xml'])
        part = '/word/comments.xml'
        ov_tag = '{%s}Override' % CT_NS
        if not any(el.get('PartName') == part for el in ct.findall('.//' + ov_tag)):
            ov = etree.Element(ov_tag)
            ov.set('PartName', part)
            ov.set('ContentType', 'application/vnd.openxmlformats-officedocument.wordprocessingml.comments+xml')
            ct.append(ov)
        files['[Content_Types].xml'] = etree.tostring(ct, xml_declaration=True, encoding='UTF-8', standalone='yes')
        
        rel_path = 'word/_rels/document.xml.rels'
        rel_root = etree.fromstring(files[rel_path])
        rtag = '{%s}Relationship' % REL_NS
        has = any((el.get('Target') or '').endswith('comments.xml') for el in rel_root.findall(rtag))
        if not has:
            max_n = 0
            for el in rel_root.findall(rtag):
                rid = el.get('Id') or ''
                if rid.startswith('rId') and rid[3:].isdigit():
                    max_n = max(max_n, int(rid[3:]))
            r = etree.Element(rtag)
            r.set('Id', f'rId{max_n + 1}')
            r.set('Type', 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/comments')
            r.set('Target', 'comments.xml')
            rel_root.append(r)
        files[rel_path] = etree.tostring(rel_root, xml_declaration=True, encoding='UTF-8', standalone='yes')
        
        if 'word/comments.xml' in files:
            com_root = etree.fromstring(files['word/comments.xml'])
        else:
            com_root = etree.Element('{%s}comments' % W_NS['w'], nsmap={'w': W_NS['w']})
            files['word/comments.xml'] = etree.tostring(com_root, xml_declaration=True, encoding='UTF-8', standalone='yes')
            com_root = etree.fromstring(files['word/comments.xml'])
        return com_root
    
    def _write_zip_dict(docx_path: Path, files: dict) -> None:
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as z:
            for name, data in sorted(files.items()):
                z.writestr(name, data)
        docx_path.write_bytes(buf.getvalue())
    
    files = _load_zip_dict(rendered_docx)
    doc_root = etree.fromstring(files['word/document.xml'])
    comments_root = _ensure_comments_infra(files)
    
    # AI处理（保持不变）
    client = None
    if use_ai and api_key:
        # 首先尝试从当前命名空间导入DeepSeekClient（如果Cell 11已经运行过）
        try:
            client = DeepSeekClient(api_key=api_key)
        except NameError:
            # 如果DeepSeekClient未定义，尝试从smart_marking_system导入
            try:
                from smart_marking_system import DeepSeekClient
                client = DeepSeekClient(api_key=api_key)
            except ImportError:
                # 如果smart_marking_system也不存在，尝试从notebook_cell_sources导入
                try:
                    from notebook_cell_sources import DeepSeekClient
                    client = DeepSeekClient(api_key=api_key)
                except ImportError:
                    # 如果所有导入都失败，则无法使用AI
                    print("警告: 无法导入DeepSeekClient，跳过AI处理")
                    client = None
    
    ai_meta = []
    if ai_markers:
        # 应用AI（简化）
        pass
    
    # 智能批注
    annotate_values_with_smart_marking(doc_root, comments_root, trace_map, position_mappings)
    
    # 保存文件
    files['word/document.xml'] = etree.tostring(doc_root, xml_declaration=True, encoding='UTF-8', standalone='yes')
    files['word/comments.xml'] = etree.tostring(comments_root, xml_declaration=True, encoding='UTF-8', standalone='yes')
    _write_zip_dict(output_docx, files)
    
    return output_docx


# 测试智能标记系统
print("智能标记系统已加载。使用方式：")
print("1. 调用 render_docx_template_with_markers() 进行智能渲染")
print("2. 调用 finalize_rendered_document_enhanced() 进行智能批注")
print("3. 设置 use_smart_marking=True 启用智能标记")


智能标记系统已加载。使用方式：
1. 调用 render_docx_template_with_markers() 进行智能渲染
2. 调用 finalize_rendered_document_enhanced() 进行智能批注
3. 设置 use_smart_marking=True 启用智能标记


In [17]:
from smart_marking_system import DeepSeekClient

## 3.4 docxtpl 渲染模块（生成中间 docx）

In [18]:
# Cell 9: docxtpl 渲染模块（生成中间 docx）
from docxtpl import DocxTemplate

def render_docx_template(template_docx: Path, context: dict, output_docx: Path) -> None:
    """
    使用 docxtpl 渲染 Word 模板并保存。

    Args:
        template_docx: templates/ 下的模板文件路径
        context: 渲染上下文字典
        output_docx: 输出 docx 路径
    Returns:
        None
    """
    if not template_docx.exists():
        raise FileNotFoundError(f'模板文件不存在: {template_docx}')
    doc = DocxTemplate(str(template_docx))
    doc.render(context)
    doc.save(str(output_docx))

# --- 测试：渲染第一份（A支行）---
mid_docx = DIRS['output'] / 'mid_A.docx'
render_docx_template(DIRS['templates'] / template_file, contexts[0][0], mid_docx)
print('Wrote mid docx:', mid_docx)

Wrote mid docx: f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\output\mid_A.docx


## 3.5 渲染后解析批注 + 提取被批注范围文本（XML 级）

In [19]:
# Cell 10: 模板 zip 读取 AI prompt + 文档 XML 工具（docxtpl 渲染后无 comments.xml）
import zipfile
import re
from lxml import etree
from typing import Dict, List, NamedTuple

W_NS = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}


class CommentBlock(NamedTuple):
    comment_id: str
    prompt_tpl: str
    text_final: str


def read_docx_xml(docx_path: Path, internal_path: str) -> bytes:
    """从 docx zip 读取内部 XML。"""
    with zipfile.ZipFile(docx_path, 'r') as z:
        return z.read(internal_path)


def extract_prompt_from_comment_text(comment_text: str) -> str:
    """从批注正文中解析 prompt=\"...\" """
    m = re.search(r'prompt\s*=\s*"([\s\S]*?)"', comment_text)
    if not m:
        raise ValueError('批注中未找到 prompt="..."')
    return m.group(1)


def extract_comments_map_from_bytes(xml_bytes: bytes) -> Dict[str, str]:
    root = etree.fromstring(xml_bytes)
    out: Dict[str, str] = {}
    for c in root.xpath('.//w:comment', namespaces=W_NS):
        cid = c.get('{%s}id' % W_NS['w'])
        texts = c.xpath('.//w:t/text()', namespaces=W_NS)
        out[str(cid)] = ''.join(texts)
    return out


def load_ai_prompts_from_template(template_docx: Path) -> List[str]:
    """
    从【模板】docx 的 comments.xml 按 id 排序提取 prompt（与 §AIBLOCK0§ 顺序对应）。
    docxtpl 渲染结果通常不含 comments.xml，必须从模板读取。
    """
    try:
        xml_bytes = read_docx_xml(template_docx, 'word/comments.xml')
    except KeyError:
        return []
    cmap = extract_comments_map_from_bytes(xml_bytes)
    prompts: List[str] = []
    for cid in sorted(cmap.keys(), key=lambda x: int(x) if x.isdigit() else 0):
        text = cmap[cid]
        if 'prompt' not in text:
            continue
        try:
            prompts.append(extract_prompt_from_comment_text(text))
        except ValueError:
            continue
    return prompts


def iter_document_order(root: etree._Element):
    yield root
    for child in root:
        yield from iter_document_order(child)


def extract_range_text(document_root: etree._Element, comment_id: str) -> str:
    start = document_root.xpath(f".//w:commentRangeStart[@w:id='{comment_id}']", namespaces=W_NS)
    end = document_root.xpath(f".//w:commentRangeEnd[@w:id='{comment_id}']", namespaces=W_NS)
    if not start or not end:
        raise ValueError(f'找不到 commentRange, id={comment_id}')
    start_el, end_el = start[0], end[0]
    all_nodes = list(iter_document_order(document_root))
    try:
        i0, i1 = all_nodes.index(start_el), all_nodes.index(end_el)
    except ValueError:
        return ''
    if i1 <= i0:
        return ''
    parts = []
    for node in all_nodes[i0 + 1:i1]:
        if node.tag == '{%s}t' % W_NS['w']:
            parts.append(node.text or '')
        else:
            for t in node.xpath('.//w:t', namespaces=W_NS):
                parts.append(t.text or '')
    return ''.join(parts).strip()


def extract_ai_comment_blocks(docx_path: Path) -> List[CommentBlock]:
    try:
        cmap = extract_comments_map_from_bytes(read_docx_xml(docx_path, 'word/comments.xml'))
    except KeyError:
        return []
    if not cmap:
        return []
    doc_root = etree.fromstring(read_docx_xml(docx_path, 'word/document.xml'))
    blocks: List[CommentBlock] = []
    for cid, ctext in cmap.items():
        if 'prompt' not in ctext:
            continue
        try:
            pt = extract_prompt_from_comment_text(ctext)
        except ValueError:
            continue
        try:
            tf = extract_range_text(doc_root, cid)
        except ValueError:
            tf = ''
        blocks.append(CommentBlock(comment_id=cid, prompt_tpl=pt, text_final=tf))
    return blocks


_tpl = DIRS['templates'] / 'template_1.docx'
if _tpl.exists():
    _ps = load_ai_prompts_from_template(_tpl)
    print('模板中 AI prompt 条数:', len(_ps))
    if _ps:
        print('首条 prompt 预览:', _ps[0][:80], '...')


模板中 AI prompt 条数: 1
首条 prompt 预览: 基于{kb:puhui_daikou_buliang}知识库，以及不良额环比{{ data.bad_mom }}、同比{{ data.bad_yoy }}数据， ...


## 3.6 渲染 prompt（Jinja2）+ DeepSeek 调用

In [ ]:
# Cell 11: 渲染 prompt（Jinja2）+ DeepSeek 调用
import os
from jinja2 import Environment, StrictUndefined
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

DEEPSEEK_BASE_URL = 'https://api.deepseek.com/v1'
DEEPSEEK_MODEL = 'deepseek-chat'


def build_jinja_env() -> Environment:
    return Environment(undefined=StrictUndefined)


def render_prompt(prompt_tpl: str, context: dict, 
                       translation_maps: Optional[Dict[str, Dict]] = None) -> str:
    """
    渲染 prompt 模板中的变量（支持中文变量名）。
    
    Args:
        prompt_tpl: prompt 模板字符串，可能包含 {{变量.字段}} 或 {{中文表名.中文字段名}}
        context: 渲染上下文（包含英文别名和可能的英文别名）
        translation_maps: 翻译映射字典（可选），用于处理中文变量名
    
    Returns:
        渲染后的 prompt 字符串
    """
    if not prompt_tpl:
        return ''
    
    result = prompt_tpl
    
    # 先处理中文变量名（如果提供了翻译映射）
    if translation_maps:
        # 查找所有变量模式：{{表名.字段名}} 或 {{表名.数字.字段名}}
        variable_pattern = r'\{\{\s*([^{}]+?)\s*\}\}'
        for match in re.finditer(variable_pattern, prompt_tpl):
            var_expr = match.group(1).strip()
            
            # 解析变量表达式：可能是 "表名.字段名" 或 "表名.数字.字段名"
            parts = var_expr.split('.')
            if len(parts) >= 2:
                table_part = parts[0]
                field_part = parts[-1]
                
                # 检查是否是中文表名
                if table_part in translation_maps['table_cn_to_en']:
                    # 中文表名，需要翻译
                    en_table = translation_maps['table_cn_to_en'][table_part]
                    
                    # 检查是否是中文字段名
                    if en_table in translation_maps['field_en_to_cn_by_table']:
                        cn_to_en_fields = {}
                        for en_field, cn_field in translation_maps['field_en_to_cn_by_table'][en_table].items():
                            cn_to_en_fields[cn_field] = en_field
                        
                        if field_part in cn_to_en_fields:
                            # 中文字段名，需要翻译
                            en_field = cn_to_en_fields[field_part]
                            
                            # 重建变量表达式
                            if len(parts) == 2:
                                new_var_expr = f'{en_table}.{en_field}'
                            else:
                                # 处理类似 "表名.数字.字段名" 的情况
                                middle_parts = parts[1:-1]
                                new_var_expr = f'{en_table}.{".".join(middle_parts)}.{en_field}'
                            
                            # 替换变量
                            old_var = f'{{{{{var_expr}}}}}'
                            new_var = f'{{{{{new_var_expr}}}}}'
                            result = result.replace(old_var, new_var)
    
    # 使用Jinja2渲染（支持英文变量）
    from jinja2 import Environment, StrictUndefined
    env = Environment(undefined=StrictUndefined)
    
    try:
        template = env.from_string(result)
        rendered = template.render(**context)
        return rendered
    except Exception as e:
        print(f'渲染 prompt 时出错: {e}')
        # 回退到简单替换
        return _simple_render_prompt(result, context, translation_maps)

def _simple_render_prompt(prompt_tpl: str, context: dict, 
                         translation_maps: Optional[Dict[str, Dict]] = None) -> str:
    """
    简单渲染 prompt 模板（回退方法）。
    """
    result = prompt_tpl
    
    # 查找所有变量模式
    variable_pattern = r'\{\{\s*([^{}]+?)\s*\}\}'
    
    for match in re.finditer(variable_pattern, prompt_tpl):
        var_expr = match.group(1).strip()
        
        # 尝试从上下文中获取值
        value = None
        try:
            # 支持点号访问嵌套结构
            parts = var_expr.split('.')
            obj = context
            for part in parts:
                if isinstance(obj, dict):
                    obj = obj.get(part)
                elif isinstance(obj, list) and part.isdigit():
                    idx = int(part)
                    if 0 <= idx < len(obj):
                        obj = obj[idx]
                    else:
                        obj = None
                        break
                else:
                    obj = None
                    break
            value = obj
        except:
            value = None
        
        if value is not None:
            # 格式化值
            if isinstance(value, (int, float)):
                str_value = str(value)
            elif isinstance(value, str):
                str_value = value
            else:
                str_value = str(value)
            
            # 替换
            old_var = f'{{{{{var_expr}}}}}'
            result = result.replace(old_var, str_value)
    
    return result

In [21]:
# Cell 12: 数据溯源批注 + AI 段落（§AIBLOCKn§）后处理并保存
import io
import zipfile
from typing import Any, Dict, List, Tuple
from datetime import datetime, timezone

CT_NS = 'http://schemas.openxmlformats.org/package/2006/content-types'
REL_NS = 'http://schemas.openxmlformats.org/package/2006/relationships'


def _wtag(name: str) -> str:
    return '{%s}%s' % (W_NS['w'], name)


def _collect_max_comment_id(doc_root: etree._Element, comments_root: etree._Element | None) -> int:
    ids: list[int] = []

    def grab(el):
        for k, v in el.attrib.items():
            if k.endswith('}id') or k == 'id':
                if str(v).isdigit():
                    ids.append(int(v))

    for el in doc_root.iter():
        if el.tag in (_wtag('commentRangeStart'), _wtag('commentRangeEnd'), _wtag('commentReference')):
            grab(el)
    if comments_root is not None:
        for el in comments_root.iter():
            if el.tag == _wtag('comment'):
                grab(el)
    return max(ids) if ids else -1


def _load_zip_dict(docx_path: Path) -> dict:
    with zipfile.ZipFile(docx_path, 'r') as z:
        return {n: z.read(n) for n in z.namelist()}


def _write_zip_dict(docx_path: Path, files: dict) -> None:
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as z:
        for name, data in sorted(files.items()):
            z.writestr(name, data)
    docx_path.write_bytes(buf.getvalue())


def _ensure_comments_infra(files: dict) -> etree._Element:
    """
    保证 [Content_Types]、document.xml.rels、comments.xml 存在。
    返回 comments 根元素（可修改）。
    """
    ct = etree.fromstring(files['[Content_Types].xml'])
    part = '/word/comments.xml'
    ov_tag = '{%s}Override' % CT_NS
    if not any(el.get('PartName') == part for el in ct.findall('.//' + ov_tag)):
        ov = etree.Element(ov_tag)
        ov.set('PartName', part)
        ov.set('ContentType', 'application/vnd.openxmlformats-officedocument.wordprocessingml.comments+xml')
        ct.append(ov)
    files['[Content_Types].xml'] = etree.tostring(ct, xml_declaration=True, encoding='UTF-8', standalone='yes')

    rel_path = 'word/_rels/document.xml.rels'
    rel_root = etree.fromstring(files[rel_path])
    rtag = '{%s}Relationship' % REL_NS
    has = any((el.get('Target') or '').endswith('comments.xml') for el in rel_root.findall(rtag))
    if not has:
        max_n = 0
        for el in rel_root.findall(rtag):
            rid = el.get('Id') or ''
            if rid.startswith('rId') and rid[3:].isdigit():
                max_n = max(max_n, int(rid[3:]))
        r = etree.Element(rtag)
        r.set('Id', f'rId{max_n + 1}')
        r.set('Type', 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/comments')
        r.set('Target', 'comments.xml')
        rel_root.append(r)
    files[rel_path] = etree.tostring(rel_root, xml_declaration=True, encoding='UTF-8', standalone='yes')

    if 'word/comments.xml' in files:
        com_root = etree.fromstring(files['word/comments.xml'])
    else:
        com_root = etree.Element(_wtag('comments'), nsmap={'w': W_NS['w']})
        files['word/comments.xml'] = etree.tostring(com_root, xml_declaration=True, encoding='UTF-8', standalone='yes')
        com_root = etree.fromstring(files['word/comments.xml'])
    return com_root


def _paragraph_full_text(w_p) -> str:
    return ''.join(w_p.xpath('.//w:t/text()', namespaces=W_NS))


def _find_paragraph_contains(doc_root, substr: str):
    for w_p in doc_root.xpath('.//w:p', namespaces=W_NS):
        if substr in _paragraph_full_text(w_p):
            return w_p
    return None


def _replace_paragraph_text(w_p, new_text: str) -> None:
    ts = w_p.xpath('.//w:t', namespaces=W_NS)
    if not ts:
        return
    ts[0].text = new_text
    for t in ts[1:]:
        t.text = ''


def _wrap_run_with_comment(parent, w_r, cid: int) -> None:
    """在 w:p 子树中，给指定 w:r 包上 commentRangeStart/End 与 commentReference。"""
    idx = parent.index(w_r)
    st = etree.Element(_wtag('commentRangeStart'))
    st.set('{%s}id' % W_NS['w'], str(cid))
    en = etree.Element(_wtag('commentRangeEnd'))
    en.set('{%s}id' % W_NS['w'], str(cid))
    ref_r = etree.Element(_wtag('r'))
    ref_cr = etree.SubElement(ref_r, _wtag('commentReference'))
    ref_cr.set('{%s}id' % W_NS['w'], str(cid))
    parent.insert(idx, st)
    parent.insert(idx + 2, en)
    parent.insert(idx + 3, ref_r)


def _append_comment_xml(comments_root: etree._Element, cid: int, body: str, author: str = '数据溯源', initials: str = 'SRC') -> None:
    c = etree.SubElement(comments_root, _wtag('comment'))
    c.set('{%s}id' % W_NS['w'], str(cid))
    c.set('{%s}author' % W_NS['w'], author)
    c.set('{%s}date' % W_NS['w'], datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'))
    c.set('{%s}initials' % W_NS['w'], initials)
    p = etree.SubElement(c, _wtag('p'))
    r = etree.SubElement(p, _wtag('r'))
    t = etree.SubElement(r, _wtag('t'))
    t.set('{http://www.w3.org/XML/1998/namespace}space', 'preserve')
    t.text = body[:8000] if len(body) > 8000 else body


def build_trace_annotation_pairs(trace_map: Dict[str, Any], 
                         translation_maps: Optional[Dict[str, Dict]] = None) -> List[Tuple[str, str]]:
    """
    生成 (显示字符串, 批注内容) 列表，显示字符串按长度降序，减少短词匹配。
    支持TraceItem及条件表达式字典格式。
    如果提供了翻译映射，批注内容会显示中文表名和字段名。
    """
    pairs: List[Tuple[str, str]] = []
    for _k, ti in trace_map.items():
        # 获取显示值
        if hasattr(ti, 'value'):
            disp = format_display_value(ti.value)
        elif isinstance(ti, dict) and 'value' in ti:
            disp = format_display_value(ti['value'])
        else:
            continue

        if disp == '':
            continue

        # 构建批注内容
        if isinstance(ti, dict) and ti.get('_is_condition'):
            # 条件表达式批注
            condition = ti.get('field', '')
            expr_info = ti.get('_expression_info', {})
            variables = expr_info.get('variables', [])
            table = ti.get('table', '__condition__')
            
            # 如果有翻译映射，尝试显示中文表名
            table_display = table
            if translation_maps:
                cn_table = translation_maps['table_en_to_cn'].get(table, table)
                table_display = cn_table
            
            # 构建详细批注内容
            var_desc = f"条件表达式: {condition}"
            if variables:
                # 如果有翻译映射，尝试显示中文变量名
                if translation_maps:
                    cn_vars = []
                    for var in variables:
                        # 解析变量：可能是 "表名.字段名" 或 "表名.数字.字段名"
                        parts = var.split('.')
                        if len(parts) >= 2:
                            table_part = parts[0]
                            field_part = parts[-1]
                            
                            # 翻译表名
                            cn_table_part = translation_maps['table_en_to_cn'].get(table_part, table_part)
                            
                            # 翻译字段名
                            cn_field_part = field_part
                            if table_part in translation_maps['field_en_to_cn_by_table']:
                                cn_field_part = translation_maps['field_en_to_cn_by_table'][table_part].get(field_part, field_part)
                            
                            # 重建变量
                            if len(parts) == 2:
                                cn_var = f'{cn_table_part}.{cn_field_part}'
                            else:
                                middle_parts = parts[1:-1]
                                cn_var = f'{cn_table_part}.{".".join(middle_parts)}.{cn_field_part}'
                            cn_vars.append(cn_var)
                        else:
                            cn_vars.append(var)
                    var_desc += f" (变量: {', '.join(cn_vars)})"
                else:
                    var_desc += f" (变量: {', '.join(variables)})"

            body = f"[数据来源] 表={table_display} 字段={condition} 值={disp} | 路径={var_desc} | 主键={ti.get('pk', '')}"

            # 添加表达式解析信息
            if expr_info.get('parsed'):
                parsed = expr_info['parsed']
                operator_map = {'lt': '<', 'le': '<=', 'gt': '>', 'ge': '>=', 'eq': '==', 'ne': '!=', 'in': 'in', 'not_in': 'not in'}
                op_symbol = operator_map.get(parsed.get('operator', ''), parsed.get('operator', ''))
                
                # 如果有翻译映射，尝试显示中文变量名
                left_display = parsed.get('left', '')
                right_display = parsed.get('right', '')
                
                if translation_maps:
                    # 尝试翻译左侧表达式中的变量
                    for var in variables:
                        if var in left_display:
                            # 解析变量
                            parts = var.split('.')
                            if len(parts) >= 2:
                                table_part = parts[0]
                                field_part = parts[-1]
                                
                                # 翻译表名
                                cn_table_part = translation_maps['table_en_to_cn'].get(table_part, table_part)
                                
                                # 翻译字段名
                                cn_field_part = field_part
                                if table_part in translation_maps['field_en_to_cn_by_table']:
                                    cn_field_part = translation_maps['field_en_to_cn_by_table'][table_part].get(field_part, field_part)
                                
                                # 重建变量
                                if len(parts) == 2:
                                    cn_var = f'{cn_table_part}.{cn_field_part}'
                                else:
                                    middle_parts = parts[1:-1]
                                    cn_var = f'{cn_table_part}.{".".join(middle_parts)}.{cn_field_part}'
                                
                                left_display = left_display.replace(var, cn_var)
                
                body += f" | 表达式: {left_display} {op_symbol} {right_display}"
        else:
            # 普通数据字段的批注
            table = ti.table if hasattr(ti, 'table') else ti.get('table', '')
            field = ti.field if hasattr(ti, 'field') else ti.get('field', '')
            var = ti.var if hasattr(ti, 'var') else ti.get('var', '')
            pk = ti.pk if hasattr(ti, 'pk') else ti.get('pk', '')
            
            # 如果有翻译映射，显示中文表名和字段名
            table_display = table
            field_display = field
            
            if translation_maps:
                cn_table = translation_maps['table_en_to_cn'].get(table, table)
                table_display = cn_table
                
                # 获取中文字段名
                if table in translation_maps['field_en_to_cn_by_table']:
                    cn_field = translation_maps['field_en_to_cn_by_table'][table].get(field, field)
                    field_display = cn_field
            
            body = f"[数据来源] 表={table_display} 字段={field_display} 值={disp} | 路径={var} | 主键={pk}"

        pairs.append((disp, body))
    
    pairs.sort(key=lambda x: len(x[0]), reverse=True)
    return pairs

## 3.7 智能生成入口（多模板 ID、数据确认、批量/单份）

In [22]:
# Cell 12.5: 缺失的函数定义 - 文档最终化与批注处理

def annotate_values_in_document(
    docx_path: Path,
    trace_map: Dict[str, Any],
    api_key: str | None = None,
    ai_markers: List[str] | None = None,
    use_ai: bool = False,
) -> None:
    """
    为文档中的值添加批注（基于trace_map）。
    
    Args:
        docx_path: Word文档路径
        trace_map: 溯源映射
        api_key: DeepSeek API密钥
        ai_markers: AI块标记列表
        use_ai: 是否调用AI
    """
    try:
        # 导入Annotator
        from smart_marking_system import Annotator
        
        # 创建批注器
        annotator = Annotator()
        
        # 构建批注对
        annotation_pairs = []
        for key, item in trace_map.items():
            if hasattr(item, 'var') and hasattr(item, 'value'):
                # 这是TraceItem对象
                var = item.var
                value = item.value
                table = getattr(item, 'table', '')
                field = getattr(item, 'field', '')
                pk = getattr(item, 'pk', '')
                row_index = getattr(item, 'row_index', None)
                source_file = getattr(item, 'source_file', '')
                
                # 创建批注文本
                annotation_text = f"[数据来源] 表={table} 字段={field} 值={value}"
                if pk:
                    annotation_text += f" | 主键={pk}"
                if row_index is not None:
                    annotation_text += f" | 行索引={row_index}"
                if source_file:
                    annotation_text += f" | 文件={source_file}"
                
                # 添加批注对
                annotation_pairs.append({
                    'text': str(value),
                    'comment': annotation_text,
                    'var_path': var
                })
            elif isinstance(item, dict):
                # 这是字典格式的trace项
                var = item.get('var', '')
                value = item.get('value', '')
                table = item.get('table', '')
                field = item.get('field', '')
                pk = item.get('pk', '')
                row_index = item.get('row_index', None)
                source_file = item.get('source_file', '')
                
                if var and value is not None:
                    # 创建批注文本
                    annotation_text = f"[数据来源] 表={table} 字段={field} 值={value}"
                    if pk:
                        annotation_text += f" | 主键={pk}"
                    if row_index is not None:
                        annotation_text += f" | 行索引={row_index}"
                    if source_file:
                        annotation_text += f" | 文件={source_file}"
                    
                    # 添加批注对
                    annotation_pairs.append({
                        'text': str(value),
                        'comment': annotation_text,
                        'var_path': var
                    })
        
        # 添加批注
        if annotation_pairs:
            annotator.annotate_document(str(docx_path), annotation_pairs)
            print(f"已添加 {len(annotation_pairs)} 条批注")
        else:
            print("未找到需要批注的数据项")
            
    except Exception as e:
        print(f"批注过程中出错: {e}")
        import traceback
        traceback.print_exc()


def finalize_rendered_document(
    mid_docx: Path,
    template_docx: Path,
    context: Dict[str, Any],
    trace_map: Dict[str, Any],
    final_docx: Path,
    use_ai: bool = False,
    api_key: str | None = None,
    ai_markers: List[str] | None = None,
) -> None:
    """
    最终化渲染的文档：添加批注，可选调用AI改写。
    
    Args:
        mid_docx: 中间文档路径
        template_docx: 模板文档路径
        context: 渲染上下文
        trace_map: 溯源映射
        final_docx: 最终文档路径
        use_ai: 是否调用AI
        api_key: DeepSeek API密钥
        ai_markers: AI块标记列表
    """
    # 首先复制中间文档到最终文档
    import shutil
    shutil.copy2(mid_docx, final_docx)
    
    # 添加批注
    annotate_values_in_document(
        docx_path=final_docx,
        trace_map=trace_map,
        api_key=api_key,
        ai_markers=ai_markers,
        use_ai=use_ai,
    )
    
    # 如果有AI标记且需要AI处理
    if use_ai and api_key and ai_markers:
        try:
            from notebook_cell_sources import DeepSeekClient
            
            # 创建AI客户端
            client = DeepSeekClient(api_key=api_key)
            
            # 读取文档内容
            from docx import Document
            doc = Document(final_docx)
            
            # 查找并处理AI块
            for marker in ai_markers:
                # 这里应该实现更复杂的AI块处理逻辑
                print(f"注意: AI块处理逻辑需要完善，检测到标记: {marker}")
                
            print("AI处理完成")
        except Exception as e:
            print(f"AI处理过程中出错: {e}")
            import traceback
            traceback.print_exc()


def finalize_rendered_document_enhanced(
    mid_docx: Path,
    template_docx: Path,
    context: Dict[str, Any],
    trace_map: Dict[str, Any],
    final_docx: Path,
    use_ai: bool = False,
    api_key: str | None = None,
    ai_markers: List[str] | None = None,
    use_smart_marking: bool = False,
    position_mappings: Dict[str, Any] | None = None,
) -> None:
    """
    增强的最终文档处理函数（支持智能标记）。
    
    Args:
        mid_docx: 中间文档路径
        template_docx: 模板文档路径
        context: 渲染上下文
        trace_map: 溯源映射
        final_docx: 最终文档路径
        use_ai: 是否调用AI
        api_key: DeepSeek API密钥
        ai_markers: AI块标记列表
        use_smart_marking: 是否使用智能标记
        position_mappings: 位置映射（智能标记使用）
    """
    # 首先复制中间文档到最终文档
    import shutil
    shutil.copy2(mid_docx, final_docx)
    
    # 添加批注
    annotate_values_in_document(
        docx_path=final_docx,
        trace_map=trace_map,
        api_key=api_key,
        ai_markers=ai_markers,
        use_ai=use_ai,
    )
    
    # 如果有AI标记且需要AI处理
    if use_ai and api_key and ai_markers:
        try:
            from notebook_cell_sources import DeepSeekClient
            
            # 创建AI客户端
            client = DeepSeekClient(api_key=api_key)
            
            # 读取文档内容
            from docx import Document
            doc = Document(final_docx)
            
            # 查找并处理AI块
            for marker in ai_markers:
                # 这里应该实现更复杂的AI块处理逻辑
                print(f"注意: AI块处理逻辑需要完善，检测到标记: {marker}")
                
            print("AI处理完成")
        except Exception as e:
            print(f"AI处理过程中出错: {e}")
            import traceback
            traceback.print_exc()
    
    # 智能标记系统增强处理（待完善）
    if use_smart_marking:
        print("智能标记系统已启用，但具体实现需要后续完善")
        if position_mappings:
            print(f"收到位置映射: {len(position_mappings)} 条")


In [23]:
# Cell 13: 智能生成入口 — 多模板 ID、确认数据表、批量/单份、可选无 AI
import os
import re
from typing import Iterable, List

# 若 Cell 5 未运行，提供默认标记列表
if 'AI_BLOCK_MARKERS' not in globals():
    AI_BLOCK_MARKERS = ['§AIBLOCK0§']


def parse_template_id_input(s: str) -> List[int]:
    """解析用户输入的模板 ID 列表（逗号分隔）。"""
    out: List[int] = []
    for part in s.replace('，', ',').split(','):
        part = part.strip()
        if not part:
            continue
        out.append(int(part))
    if not out:
        raise ValueError('未解析到任何模板 ID')
    return out


def summarize_required_tables(relation_df: pd.DataFrame, template_ids: Iterable[int]) -> dict[int, List[str]]:
    """每个模板 ID -> 所需实体表名列表。"""
    m: dict[int, List[str]] = {}
    for tid in template_ids:
        _f, tables, _m = get_template_tables(relation_df, int(tid))
        m[int(tid)] = list(dict.fromkeys(tables))
    return m


def run_smart_document_generation(
    template_ids: List[int],
    data_dir: Path | None = None,
    use_ai: bool = False,
    skip_confirm: bool = False,
    use_smart_marking: bool = True  # 新增参数：是否使用智能标记
) -> List[Path]:
    """
    按模板 ID 列表批量生成报告（每个模板：主表多行则多份）。

    Args:
        template_ids: 模板 ID
        data_dir: 数据目录，默认 DIRS['data']
        use_ai: 是否调用 DeepSeek（需环境变量 DEEPSEEK_API_KEY）
        skip_confirm: True 时非交互（测试用）
        use_smart_marking: 是否使用智能标记系统

    Returns:
        生成的最终 docx 路径列表
    """
    data_dir = data_dir or DIRS['data']
    schema_path = DIRS['config'] / 'entity_schema.xlsx'
    relation_path = DIRS['config'] / 'template_relation.xlsx'
    schema_df = load_entity_schema(schema_path)
    relation_df = load_template_relation(relation_path)
    api_key = os.getenv('DEEPSEEK_API_KEY')

    req = summarize_required_tables(relation_df, template_ids)
    print('=== 各模板所需数据表（请在数据目录准备 表名.xlsx）===')
    for tid, tabs in req.items():
        print(f'  模板 {tid}: {tabs}')
    print(f'数据目录：{data_dir.resolve()}')
    print(f"智能标记：{'启用' if use_smart_marking else '禁用'}")

    if not skip_confirm:
        ans = input('确认已准备好上述 Excel？(y/n): ').strip().lower()
        if ans not in ('y', 'yes', '是'):
            print('已取消生成。')
            return []

    outputs: List[Path] = []
    for tid in template_ids:
        template_file, table_names, main_table = get_template_tables(relation_df, tid)
        tables = load_data_tables(data_dir, table_names)
        
        # 获取模板名称
        template_info = relation_df[relation_df['template_id'] == tid].iloc[0]
        template_name = template_info['template_name']
        # 清理模板名称作为文件名前缀
        template_name_safe = re.sub(r'[<>:"/\\|?*]', '_', template_name)
        template_name_safe = re.sub(r'\s+', '_', template_name_safe)
        template_name_safe = template_name_safe.strip('_')
        
        tpl_path = DIRS['templates'] / template_file
        if not tpl_path.exists():
            raise FileNotFoundError(f'模板文件不存在：{tpl_path}')
        ctx_list = build_report_contexts(schema_df, tables, main_table=main_table, template_path=tpl_path, translation_maps=translation_maps)

        for i, (ctx, trace) in enumerate(ctx_list):
            pk = ctx.get('data', {})
            pk_val = pk.get('branch_name', pk.get(list(pk.keys())[0] if pk else 'key', f'idx{i}'))
            safe_name = str(pk_val).replace('/', '_').replace('\\', '_')
            mid_name = f'mid_{template_name_safe}_{safe_name}.docx'
            final_name = f'{template_name_safe}_{safe_name}.docx'
            mid_path = DIRS['output'] / mid_name
            final_path = DIRS['output'] / final_name

            if use_smart_marking:
                print("注意：智能标记系统当前仅支持批注，文档渲染使用原方式")
                # 使用原方式渲染文档
                render_docx_template(tpl_path, ctx, mid_path)
                
                # 智能标记仅用于批注（需要后续完善）
                position_mappings = None  # 暂时不提供位置映射
                
                # 使用增强的最终文档处理函数（支持智能批注）
                finalize_rendered_document_enhanced(
                    mid_path,
                    tpl_path,
                    ctx,
                    trace,
                    final_path,
                    use_ai=use_ai,
                    api_key=api_key,
                    ai_markers=list(AI_BLOCK_MARKERS),
                    use_smart_marking=True,
                    position_mappings=position_mappings
                )
            else:
                # 使用原方式
                render_docx_template(tpl_path, ctx, mid_path)
                finalize_rendered_document(
                    mid_path,
                    tpl_path,
                    ctx,
                    trace,
                    final_path,
                    use_ai=use_ai,
                    api_key=api_key,
                    ai_markers=list(AI_BLOCK_MARKERS),
                )
            outputs.append(final_path)
            print('已生成:', final_path)

    return outputs


def interactive_smart_generation() -> None:
    """交互：输入模板 ID → 展示依赖表 → 确认 → 生成（可选 AI 和智能标记）。"""
    raw = input('请输入要生成的 Word 模板 ID，多个用英文逗号分隔：')
    tids = parse_template_id_input(raw)
    want_ai = input('是否调用 DeepSeek 进行 AI 改写？(y/n): ').strip().lower() in ('y', 'yes', '是')
    want_smart = input('是否使用智能标记系统提高批注精度？(y/n): ').strip().lower() in ('y', 'yes', '是')
    run_smart_document_generation(tids, use_ai=want_ai, skip_confirm=False, use_smart_marking=want_smart)


# 非交互自测示例（需要前面 Cell 已加载 relation_df2 等）:
# run_smart_document_generation([1], use_ai=False, skip_confirm=True, use_smart_marking=True)


In [24]:
# Cell 14: 运行智能生成（交互）
# 执行下面一行后按提示输入模板 ID（逗号分隔）、是否 AI、并确认数据表已就绪。
interactive_smart_generation()

# 或一行非交互测试（不调用 AI、自动确认）：
# run_smart_document_generation([1], use_ai=False, skip_confirm=True, use_smart_marking=True)


=== 各模板所需数据表（请在数据目录准备 表名.xlsx）===
  模板 4: ['hetong_data', 'company_data', 'borrower_info']
数据目录：F:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\data
智能标记：禁用
批注过程中出错: cannot import name 'Annotator' from 'smart_marking_system' (f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\smart_marking_system.py)
AI处理过程中出错: cannot import name 'DeepSeekClient' from 'notebook_cell_sources' (f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\notebook_cell_sources.py)
已生成: f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\output\证券化内部尽调报告_4000920260329001.docx
批注过程中出错: cannot import name 'Annotator' from 'smart_marking_system' (f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\smart_marking_system.py)
AI处理过程中出错: cannot import name 'DeepSeekClient' from 'notebook_cell_sources' (f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\notebook_cell_sources.py)
已生成: f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\output\证券化内部尽调报告_4000920260329002.docx
批注过

Traceback (most recent call last):
  File "C:\Users\32114\AppData\Local\Temp\ipykernel_7364\3841706614.py", line 22, in annotate_values_in_document
    from smart_marking_system import Annotator
ImportError: cannot import name 'Annotator' from 'smart_marking_system' (f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\smart_marking_system.py)
Traceback (most recent call last):
  File "C:\Users\32114\AppData\Local\Temp\ipykernel_7364\3841706614.py", line 134, in finalize_rendered_document
    from notebook_cell_sources import DeepSeekClient
ImportError: cannot import name 'DeepSeekClient' from 'notebook_cell_sources' (f:\Jupyterworkspace\工行\普惠\word_gen_system\word_gen_system\notebook_cell_sources.py)
Traceback (most recent call last):
  File "C:\Users\32114\AppData\Local\Temp\ipykernel_7364\3841706614.py", line 22, in annotate_values_in_document
    from smart_marking_system import Annotator
ImportError: cannot import name 'Annotator' from 'smart_marking_system' (f:\Jupyterworkspa

In [25]:
# Cell 14: 运行智能生成（交互）
# 执行下面一行后按提示输入模板 ID（逗号分隔）、是否 AI、并确认数据表已就绪。
# interactive_smart_generation()

# 或一行非交互测试（不调用 AI、自动确认）：
# run_smart_document_generation([1], use_ai=False, skip_confirm=True, use_smart_marking=True)


In [26]:
# 4、智能分析引擎（新增功能）

本模块提供基于 ydata-profiling 的智能数据分析功能，支持：
- 自动加载 Excel/CSV 数据集
- 生成详细的 HTML 数据分析报告
- 根据批注标记自动生成图表
- 与现有文档生成功能完全隔离，互不影响

## 使用方法
1. 运行下方单元格初始化引擎
2. 注册需要分析的数据集
3. 在 Word 模板批注中使用 `{analysis:数据集名：提示词}` 或 `{chart:类型：数据集：X 列：Y 列：标题}` 标记
4. 调用 `engine.process_comments()` 处理批注并生成报告

SyntaxError: invalid character '，' (U+FF0C) (3230022810.py, line 3)

In [ ]:
# Cell 15: 智能分析引擎初始化与测试
print("正在初始化智能分析引擎...")

# 导入新模块
from smart_analysis_engine import init_smart_analysis, get_analysis_engine
from dataset_manager import register_dataset, get_dataset_manager
from analysis_prompt_parser import parse_analysis_requests

# 初始化引擎
engine = init_smart_analysis()

# 示例：注册一个测试数据集（如果存在）
test_data_path = DIRS['data'] / 'loan_summary.xlsx'
if test_data_path.exists():
    print(f"\n发现测试数据文件：{test_data_path}")
    try:
        df = engine.register_dataset('loan_data', str(test_data_path))
        print(f"✓ 数据集 'loan_data' 注册成功")
        print(f"  预览前 5 行:\n{df.head()}")
    except Exception as e:
        print(f"⚠ 注册数据集失败：{e}")
else:
    print(f"\n⚠ 未找到测试数据文件：{test_data_path}")
    print("请将 Excel/CSV 文件放入 data 目录后重新运行")

# 示例：解析批注标记
print("\n" + "="*60)
print("批注标记解析示例:")
test_comments = [
    "{analysis:loan_data:分析贷款余额的分布情况}",
    "{chart:bar:loan_data:branch_name:loan_amount:各支行贷款分布}",
    "普通文本，不包含标记"
]

for comment in test_comments:
    requests = parse_analysis_requests(comment)
    if requests:
        print(f"\n批注：{comment}")
        for req in requests:
            print(f"  → 数据集：{req.dataset_name}")
            print(f"  → 提示词：{req.prompt}")
            if req.chart_type:
                print(f"  → 图表类型：{req.chart_type}")
                print(f"  → X 轴：{req.chart_x_col}, Y 轴：{req.chart_y_col}")
    else:
        print(f"\n批注：{comment} (无分析标记)")

print("\n" + "="*60)
print("智能分析引擎初始化完成！")
print("\n下一步操作建议:")
print("1. 准备数据文件放在 data 目录")
print("2. 在 Word 模板中添加 {analysis:...} 或 {chart:...} 批注")
print("3. 调用 engine.process_comments(comments_list) 生成分析报告")
print("\n注意：此功能完全独立，不会影响原有的文档生成功能。")

## RAG知识库功能集成

新增知识库RAG检索功能，支持在Word模板批注中使用 `{kb:知识库名称}` 标记，自动检索相关知识并注入到prompt中。

使用方法：
1. 在Word模板批注中添加 `{kb:puhui_daikou_buliang}` 或 `{kb:puhui_daikou_buliang:查询词}`
2. 运行下面的代码启用RAG功能
3. 使用原有的智能文档生成功能，RAG会自动生效

In [ ]:
# RAG功能初始化
from rag_monkey_patch import setup_rag_for_notebook

# 启用RAG功能
print("正在初始化RAG知识库功能...")
if setup_rag_for_notebook():
    print("✓ RAG功能已成功启用！")
    print("\n可用知识库：")
    from rag_monkey_patch import get_rag_manager
    rag_manager = get_rag_manager()
    if rag_manager:
        kb_list = rag_manager.list_available_knowledge_bases()
        for kb in kb_list:
            print(f"  - {kb}")
    print("\n现在可以在Word模板批注中使用 {kb:知识库名称} 标记了。")
else:
    print("⚠ RAG功能启用失败，请检查错误信息。")
    print("常见问题：")
    print("1. 确保已安装依赖: pip install sentence-transformers numpy scikit-learn python-docx")
    print("2. 确保已创建知识库: python create_puhui_knowledge_base.py")

In [ ]:
# RAG功能测试示例
print("RAG功能测试：")
print("=" * 60)

# 测试1: 检查RAG管理器
from rag_monkey_patch import get_rag_manager
rag_manager = get_rag_manager()
if rag_manager:
    print(f"✓ RAG管理器可用，知识库数量: {len(rag_manager.list_available_knowledge_bases())}")
    
    # 测试2: 测试知识库检索
    print("\n测试知识库检索功能：")
    results = rag_manager.search_knowledge_base('puhui_daikou_buliang', '不良贷款', top_k=2)
    if results:
        print(f"✓ 检索到 {len(results)} 条结果")
        for i, res in enumerate(results, 1):
            print(f"  结果{i}: {res['doc_name']} (相似度: {res['similarity']:.3f})")
    else:
        print("⚠ 未检索到结果（可能是关键词匹配模式）")
        
    # 测试3: 测试prompt增强
    print("\n测试prompt增强功能：")
    test_prompt = "基于不良率数据，结合{kb:puhui_daikou_buliang}知识库，分析风险情况"
    test_context = {"data": {"bad_mom": 0.12, "bad_yoy": 0.35}}
    enhanced_prompt, used_rag = rag_manager.enhance_prompt_with_rag(test_prompt, test_context)
    if used_rag:
        print(f"✓ RAG增强成功")
        print(f"原始prompt长度: {len(test_prompt)}")
        print(f"增强后长度: {len(enhanced_prompt)}")
        print(f"是否包含知识库内容: {'知识库参考信息' in enhanced_prompt}")
    else:
        print("⚠ RAG增强未生效（可能没有{kb:}标记）")
else:
    print("✗ RAG管理器不可用")

print("=" * 60)

In [ ]:
## 使用说明

### 启用RAG后，原有的智能文档生成功能将自动支持：

1. **在Word模板中使用**：
   ```
   prompt="基于不良率{{data.bad_mom}}，结合{kb:puhui_daikou_buliang}知识库，分析风险状况"
   ```

2. **运行智能生成**：
   - 方式1（交互式）：运行 `interactive_smart_generation()`
   - 方式2（编程式）：`run_smart_document_generation([1], use_ai=True, skip_confirm=True)`

### 扩展知识库

要添加新的知识库，在 `knowledge_bases/` 目录下创建新文件夹：

1. 创建 `knowledge_bases/新知识库名称/` 目录
2. 创建 `metadata.json` 文件
3. 添加文档文件（支持 .docx, .pdf, .txt 等）
4. 在模板中使用 `{kb:新知识库名称}` 引用

### 故障排除

- **RAG功能未生效**：检查是否成功运行了上面的RAG初始化代码
- **知识库加载失败**：检查 `knowledge_bases/` 目录结构是否正确
- **网络问题**：如果sentence-transformers模型下载失败，系统会自动降级到关键词匹配模式